<a href="https://colab.research.google.com/github/David2204269/RAHCE/blob/v2f/ffmpeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from pathlib import Path

RAR_PATH = Path("/content/drive/MyDrive/hmdb51.rar")

VIDEOS_DIR = Path("/content/HMDB51_videos")
FRAMES_DIR = Path("/content/HMDB51_frames")

# None = todos los frames
# 5 = cinco frames por segundo
FPS = 8

JPEG_QUALITY = 2
WORKERS = 2

if not RAR_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {RAR_PATH}")

print("Archivo encontrado:")
print(RAR_PATH)

Archivo encontrado:
/content/drive/MyDrive/hmdb51.rar


In [ ]:
import shutil
import subprocess

if VIDEOS_DIR.exists():
    shutil.rmtree(VIDEOS_DIR)

if FRAMES_DIR.exists():
    shutil.rmtree(FRAMES_DIR)

VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

resultado = subprocess.run(
    [
        "unrar",
        "x",
        "-o+",
        str(RAR_PATH),
        str(VIDEOS_DIR) + "/"
    ],
    capture_output=True,
    text=True
)

if resultado.returncode != 0:
    print(resultado.stdout)
    print(resultado.stderr)
    raise RuntimeError("No se pudo descomprimir el archivo RAR.")

print("Dataset descomprimido correctamente en:")
print(VIDEOS_DIR)

Dataset descomprimido correctamente en:
/content/HMDB51_videos


In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {
    ".avi", ".mp4", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

videos = sorted(
    archivo
    for archivo in VIDEOS_DIR.rglob("*")
    if archivo.is_file()
    and archivo.suffix.lower() in VIDEO_EXTENSIONS
)

print(f"Videos encontrados: {len(videos)}")

if not videos:
    raise RuntimeError(
        "No se encontraron videos. Revisa la estructura del archivo RAR."
    )


def convertir_video(video_path):
    relative_path = video_path.relative_to(VIDEOS_DIR)

    output_dir = (
        FRAMES_DIR
        / relative_path.parent
        / video_path.stem
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    output_pattern = output_dir / "frame_%06d.jpg"

    command = [
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel",
        "error",
        "-i",
        str(video_path),
    ]

    if FPS is not None:
        command.extend([
            "-vf",
            f"fps={FPS}"
        ])

    command.extend([
        "-q:v",
        str(JPEG_QUALITY),
        str(output_pattern)
    ])

    result = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        return {
            "ok": False,
            "video": str(video_path),
            "error": result.stderr
        }

    cantidad_frames = len(list(output_dir.glob("frame_*.jpg")))

    return {
        "ok": True,
        "video": str(video_path),
        "frames": cantidad_frames
    }


resultados = []

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    tareas = [
        executor.submit(convertir_video, video)
        for video in videos
    ]

    for tarea in tqdm(
        as_completed(tareas),
        total=len(tareas),
        desc="Convirtiendo videos"
    ):
        resultados.append(tarea.result())


correctos = [r for r in resultados if r["ok"]]
errores = [r for r in resultados if not r["ok"]]

total_frames = sum(r["frames"] for r in correctos)

print("\nConversión terminada")
print(f"Videos correctos: {len(correctos)}")
print(f"Videos con error: {len(errores)}")
print(f"Total de frames: {total_frames}")

if errores:
    print("\nPrimeros errores:")

    for error in errores[:10]:
        print(error["video"])
        print(error["error"][:300])
        print("-" * 50)

Videos encontrados: 6766


Convirtiendo videos:   0%|          | 0/6766 [00:00<?, ?it/s]


Conversión terminada
Videos correctos: 6766
Videos con error: 0
Total de frames: 165662


In [ ]:
import shutil
from pathlib import Path

ZIP_BASE = Path("/content/drive/MyDrive/HMDB51_frames")
ZIP_OUTPUT = Path("/content/drive/MyDrive/HMDB51_frames.zip")

if ZIP_OUTPUT.exists():
    ZIP_OUTPUT.unlink()

print("Creando ZIP en Google Drive...")

shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir="/content",
    base_dir="HMDB51_frames"
)

size_gb = ZIP_OUTPUT.stat().st_size / (1024 ** 3)

print("Archivo creado:")
print(ZIP_OUTPUT)
print(f"Tamaño: {size_gb:.2f} GB")

Creando ZIP en Google Drive...
Archivo creado:
/content/drive/MyDrive/HMDB51_frames.zip
Tamaño: 2.47 GB
